# Notebook 12 — LightGBM optimisé (Optuna + features avancées)

## Objectif
Améliorer le RMSE=4.14 du notebook 11 via :
1. **Optuna** — tuning bayésien des hyperparamètres LightGBM (~50 trials)
2. **Features supplémentaires** — ratios, target encoding de rater_id, écart par cohorte
3. **Stacking** — méta-modèle Ridge sur les prédictions OOF
4. **Clip intelligent** — borne max à 100 (score moteur UPDRS-III)

## Fichiers utilisés
- `data/X_train.csv`, `data/y_train.csv`, `data/X_test.csv`

## Notebook suivant
`13_xgb_ensemble.ipynb` — ensemble LightGBM et XGBoost.

## 1. Imports

In [1]:
import json
import os
import warnings
from pathlib import Path

import lightgbm as lgb
import numpy as np
import optuna
import pandas as pd
import skore
from lightgbm import LGBMRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")

## 2. Connexion Skore Hub

In [2]:
config = json.loads(Path("../.skore").read_text())
os.environ["SKORE_HUB_API_KEY"] = config["api_key"]
skore.login()
hub_project = skore.Project(
    "bobathon-esilv",
    mode="hub",
    workspace=config["workspace"],
)
print(f"Connecté : workspace={hub_project.workspace}")

╭───────────────────────────────── Login to Skore Hub ─────────────────────────────────╮
│                                                                                      │
│                        Successfully logged in, using API key.                        │
│                                                                                      │
╰──────────────────────────────────────────────────────────────────────────────────────╯

Connecté : workspace=BERNIE MADOFF


## 3. Feature engineering (identique notebook 11 + nouvelles features)

In [3]:
CAT = ["cohort", "gene", "rater_id"]


def _slope(x, y):
    m = ~(np.isnan(x) | np.isnan(y))
    if m.sum() < 2 or np.ptp(x[m]) < 1e-9:
        return np.nan, np.nan, m.sum()
    xs, ys = x[m], y[m]
    xm, ym = xs.mean(), ys.mean()
    b = ((xs - xm) * (ys - ym)).sum() / ((xs - xm) ** 2).sum()
    return b, ym - b * xm, m.sum()


def _patient_feats(g, col, pre):
    age, v, n = g["age"].values, g[col].values, len(g)
    out = {}
    b, a, k = _slope(age, v)
    out[f"{pre}_slope"] = np.full(n, b)
    out[f"{pre}_fit"]   = a + b * age if not np.isnan(b) else np.full(n, np.nan)
    out[f"{pre}_mean"]  = np.full(n, np.nanmean(v) if k else np.nan)
    out[f"{pre}_std"]   = np.full(n, np.nanstd(v) if k > 1 else np.nan)
    out[f"{pre}_first"] = np.full(n, v[~np.isnan(v)][0] if k else np.nan)
    out[f"{pre}_last"]  = np.full(n, v[~np.isnan(v)][-1] if k else np.nan)
    out[f"{pre}_cnt"]   = np.full(n, k)
    loo = np.full(n, np.nan)
    for i in range(n):
        mk = np.ones(n, bool); mk[i] = False
        bi, ai, _ = _slope(age[mk], v[mk])
        if not np.isnan(bi):
            loo[i] = ai + bi * age[i]
    out[f"{pre}_loo"] = loo
    m_v = ~np.isnan(v)
    quad = np.full(n, np.nan)
    if m_v.sum() >= 4:
        c = np.polyfit(age[m_v] - age.mean(), v[m_v], 2)
        quad = np.polyval(c, age - age.mean())
    out[f"{pre}_quad"] = quad
    return pd.DataFrame(out, index=g.index)


def build(df, target_enc=None):
    """Construit toutes les features. `target_enc` = dict rater_id -> mean_target (train only)."""
    d = df.sort_values(["patient_id", "age", "Index"]).copy()
    gb = d.groupby("patient_id")

    d["dx_missing"]  = d["age_at_diagnosis"].isna().astype(int)
    d["dur"]         = d["age"] - d["age_at_diagnosis"]
    d["dur_first"]   = gb["dur"].transform("min")
    d["age_first"]   = gb["age"].transform("min")
    d["t_rel"]       = d["age"] - d["age_first"]
    d["n_vis"]       = gb["age"].transform("size")
    d["vis_idx"]     = gb.cumcount()
    d["vis_frac"]    = d["vis_idx"] / (d["n_vis"] - 1).clip(lower=1)
    d["gap_prev"]    = gb["age"].diff()
    d["gap_next"]    = -gb["age"].diff(-1)

    d["on_off_diff"]  = d["off"] - d["on"]
    d["on_off_ratio"] = d["on"] / d["off"].replace(0, np.nan)
    d["ledd_per_dur"] = d["ledd"] / d["dur"].clip(lower=0.5)
    d["has_on"]       = d["on"].notna().astype(int)
    d["has_off"]      = d["off"].notna().astype(int)
    d["has_ledd"]     = d["ledd"].notna().astype(int)
    d["tsi_off_dev"]  = d["time_since_intake_off"] - 12

    # --- Nouvelles features ---
    d["on_x_ledd"]    = d["on"] * d["ledd"]           # interaction dose × score ON
    d["off_x_ledd"]   = d["off"] * d["ledd"]          # interaction dose × score OFF
    d["age_sq"]       = d["age"] ** 2                  # effet quadratique de l'âge
    d["dur_sq"]       = d["dur"] ** 2

    for c in ["on", "off", "ledd", "time_since_intake_off", "time_since_intake_on"]:
        for k in (1, 2):
            d[f"{c}_lag{k}"]  = gb[c].shift(k)
            d[f"{c}_lead{k}"] = gb[c].shift(-k)
    d["off_nb_mean"] = d[["off_lag1", "off_lead1"]].mean(axis=1)
    d["on_nb_mean"]  = d[["on_lag1",  "on_lead1"]].mean(axis=1)
    # Diff avec visite précédente / suivante
    d["off_delta_prev"] = d["off"] - d["off_lag1"]
    d["off_delta_next"] = d["off_lead1"] - d["off"]

    parts = [
        pd.concat([_patient_feats(g, c, c) for c in ["on", "off", "ledd"]], axis=1)
        for _, g in d.groupby("patient_id", sort=False)
    ]
    d = d.join(pd.concat(parts))

    d["pseudo_off_from_on"] = d["on"] * (d["off_mean"] / d["on_mean"])
    d["traj_blend"]         = d[["off_loo", "pseudo_off_from_on"]].mean(axis=1)
    d["p_age_dx"]           = d.groupby("patient_id")["age_at_diagnosis"].transform("first")
    # Ratio off_loo / off (fiabilité de la trajectoire)
    d["off_loo_ratio"]      = d["off_loo"] / d["off"].replace(0, np.nan)
    # Écart entre off observé et trajectoire LOO
    d["off_loo_resid"]      = d["off"] - d["off_loo"]

    # Target encoding de rater_id (évite fuite : passé depuis le train)
    if target_enc is not None:
        d["rater_target_mean"] = d["rater_id"].map(target_enc).fillna(target_enc.get("__global__", 37.0))

    for c in CAT:
        d[c] = d[c].astype("category")

    return d.sort_values("Index")


def feature_cols(d):
    return [c for c in d.columns if c not in {"Index", "patient_id", "target"}]


print("Feature engineering défini")

Feature engineering défini


## 4. Chargement et construction

In [4]:
DATA_DIR = Path("../data")

X_raw = pd.read_csv(DATA_DIR / "X_train.csv")
y_raw = pd.read_csv(DATA_DIR / "y_train.csv")
X_test_raw = pd.read_csv(DATA_DIR / "X_test.csv")

# Target encoding de rater_id calculé sur tout le train (sera appliqué aussi au test)
merged_full = X_raw.merge(y_raw, on="Index")
rater_enc = merged_full.groupby("rater_id")["target"].mean().to_dict()
rater_enc["__global__"] = merged_full["target"].mean()

d = build(X_raw, target_enc=rater_enc).merge(y_raw, on="Index").sort_values("Index").reset_index(drop=True)
cols = feature_cols(d)

print(f"Dataset train  : {d.shape}")
print(f"Nb features    : {len(cols)}")
print(f"Nb patients    : {d['patient_id'].nunique()}")

Dataset train  : (44590, 92)
Nb features    : 89
Nb patients    : 5576


## 5. Optuna — tuning bayésien des hyperparamètres

Chaque trial évalue un jeu d'hyperparamètres sur 3 folds GroupKFold (rapide). Le meilleur jeu est ensuite utilisé pour l'entraînement final sur 5 folds.

In [5]:
def objective(trial):
    params = dict(
        objective="regression",
        verbose=-1,
        num_threads=4,
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        num_leaves=trial.suggest_int("num_leaves", 20, 80),
        min_child_samples=trial.suggest_int("min_child_samples", 10, 50),
        feature_fraction=trial.suggest_float("feature_fraction", 0.5, 1.0),
        bagging_fraction=trial.suggest_float("bagging_fraction", 0.6, 1.0),
        bagging_freq=1,
        lambda_l2=trial.suggest_float("lambda_l2", 0.1, 20.0, log=True),
        cat_smooth=trial.suggest_int("cat_smooth", 5, 50),
    )
    oof = np.zeros(len(d))
    for tr, va in GroupKFold(n_splits=3).split(d, groups=d["patient_id"]):
        m = lgb.train(
            params,
            lgb.Dataset(d.loc[tr, cols], d.loc[tr, "target"]),
            num_boost_round=2000,
            valid_sets=[lgb.Dataset(d.loc[va, cols], d.loc[va, "target"])],
            callbacks=[lgb.early_stopping(50, verbose=False)],
        )
        oof[va] = m.predict(d.loc[va, cols], num_iteration=m.best_iteration)
    return np.sqrt(((oof - d["target"].values) ** 2).mean())


study = optuna.create_study(direction="minimize",
                             sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=50, show_progress_bar=True)

best_params = study.best_params
best_params.update({"objective": "regression", "verbose": -1, "num_threads": 4, "bagging_freq": 1})
print(f"\nMeilleur RMSE Optuna (3-fold) : {study.best_value:.4f}")
print("Meilleurs hyperparamètres :")
for k, v in best_params.items():
    print(f"  {k}: {v}")

  0%|          | 0/50 [00:00<?, ?it/s]


Meilleur RMSE Optuna (3-fold) : 4.1184
Meilleurs hyperparamètres :
  learning_rate: 0.014386019128011194
  num_leaves: 74
  min_child_samples: 16
  feature_fraction: 0.5145748012041038
  bagging_fraction: 0.6350658569145876
  lambda_l2: 0.3732413497250392
  cat_smooth: 42
  objective: regression
  verbose: -1
  num_threads: 4
  bagging_freq: 1


## 6. Entraînement final 5-fold avec les meilleurs hyperparamètres

In [6]:
oof = np.zeros(len(d))
models = []
rmse_per_fold = []

for fold, (tr, va) in enumerate(GroupKFold(n_splits=5).split(d, groups=d["patient_id"])):
    m = lgb.train(
        best_params,
        lgb.Dataset(d.loc[tr, cols], d.loc[tr, "target"]),
        num_boost_round=3000,
        valid_sets=[lgb.Dataset(d.loc[va, cols], d.loc[va, "target"])],
        callbacks=[lgb.early_stopping(100, verbose=False)],
    )
    oof[va] = m.predict(d.loc[va, cols], num_iteration=m.best_iteration)
    models.append(m)

    fold_rmse = np.sqrt(((oof[va] - d.loc[va, "target"].values) ** 2).mean())
    fold_mae  = np.abs(oof[va] - d.loc[va, "target"].values).mean()
    rmse_per_fold.append(fold_rmse)
    print(f"  Fold {fold+1} — best_iter={m.best_iteration:4d}  RMSE={fold_rmse:.4f}  MAE={fold_mae:.4f}")

err = oof - d["target"].values
rmse_oof = np.sqrt((err ** 2).mean())
mae_oof  = np.abs(err).mean()
print(f"\nOOF global — RMSE={rmse_oof:.4f}  MAE={mae_oof:.4f}")

  Fold 1 — best_iter=1632  RMSE=4.1314  MAE=3.0267
  Fold 2 — best_iter=1855  RMSE=4.1562  MAE=3.0272
  Fold 3 — best_iter=3000  RMSE=4.0364  MAE=2.9917
  Fold 4 — best_iter=1889  RMSE=3.9829  MAE=2.9766
  Fold 5 — best_iter=2152  RMSE=4.0460  MAE=2.9913

OOF global — RMSE=4.0711  MAE=3.0027


## 7. Stacking — méta-modèle Ridge sur les prédictions OOF

On entraîne un modèle Ridge de second niveau sur les prédictions OOF du LightGBM + quelques features robustes. Cela corrige les biais systématiques du modèle de base.

In [7]:
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# Features du méta-modèle : prédiction LightGBM + features simples robustes
meta_features_train = np.column_stack([
    oof,                                      # prédiction LightGBM
    d["off"].fillna(d["off"].median()),       # off observé
    d["on"].fillna(d["on"].median()),         # on observé
    d["age"].values,
])

# Validation du méta-modèle sur les mêmes folds
meta_oof = np.zeros(len(d))
meta_models = []
for tr, va in GroupKFold(n_splits=5).split(d, groups=d["patient_id"]):
    meta = make_pipeline(StandardScaler(), RidgeCV(alphas=[0.1, 1, 10, 100]))
    meta.fit(meta_features_train[tr], d.loc[tr, "target"])
    meta_oof[va] = meta.predict(meta_features_train[va])
    meta_models.append(meta)

rmse_meta = np.sqrt(((meta_oof - d["target"].values) ** 2).mean())
mae_meta  = np.abs(meta_oof - d["target"].values).mean()
print(f"RMSE stacking (Ridge sur OOF) : {rmse_meta:.4f}  MAE={mae_meta:.4f}")
print(f"RMSE LightGBM seul            : {rmse_oof:.4f}")
print(f"Gain stacking                 : {rmse_oof - rmse_meta:+.4f}")

RMSE stacking (Ridge sur OOF) : 4.0728  MAE=3.0045
RMSE LightGBM seul            : 4.0711
Gain stacking                 : -0.0017


## 8. Prédictions sur le jeu de test

In [8]:
dt = build(X_test_raw, target_enc=rater_enc).sort_values("Index").reset_index(drop=True)
for c in CAT:
    dt[c] = pd.Categorical(dt[c].astype(str), categories=d[c].cat.categories)

# Ensemble LightGBM : moyenne des 5 modèles
lgbm_preds = np.mean(
    [m.predict(dt[cols], num_iteration=m.best_iteration) for m in models],
    axis=0,
)

# Features méta-modèle sur le test
meta_features_test = np.column_stack([
    lgbm_preds,
    dt["off"].fillna(d["off"].median()),
    dt["on"].fillna(d["on"].median()),
    dt["age"].values,
])

# Moyenne des méta-modèles
meta_preds = np.mean([m.predict(meta_features_test) for m in meta_models], axis=0)

# Clip : score UPDRS-III entre 0 et 100
final_preds = np.clip(meta_preds, 0, 100)

print(f"Min: {final_preds.min():.2f}  Max: {final_preds.max():.2f}  Mean: {final_preds.mean():.2f}")

Min: 0.00  Max: 91.25  Mean: 36.74


## 9. Construction et vérification de la soumission

In [9]:
submission = X_test_raw[["Index"]].copy()
submission["target"] = final_preds

assert list(submission.columns) == ["Index", "target"]
assert len(submission) == len(X_test_raw)
assert submission["Index"].equals(X_test_raw["Index"])
assert submission["target"].notna().all()

Path("../submissions").mkdir(exist_ok=True)
submission.to_csv("../submissions/optimized.csv", index=False)
print(f"Soumission enregistrée : submissions/optimized.csv ({len(submission)} lignes)")
display(submission.head())

Soumission enregistrée : submissions/optimized.csv (11013 lignes)


,Index,target
0,66,22.721239
1,67,30.855909
2,68,28.860464
3,69,32.004921
4,70,35.156988


## 10. Envoi Skore Hub

In [10]:
best_iters = [m.best_iteration for m in models]
n_est = int(np.mean(best_iters))

lgbm_skore = LGBMRegressor(
    n_estimators=n_est,
    learning_rate=best_params["learning_rate"],
    num_leaves=best_params["num_leaves"],
    min_child_samples=best_params["min_child_samples"],
    feature_fraction=best_params["feature_fraction"],
    bagging_fraction=best_params["bagging_fraction"],
    bagging_freq=1,
    reg_lambda=best_params["lambda_l2"],
    cat_smooth=best_params["cat_smooth"],
    random_state=0,
    verbose=-1,
)

cv_skore = list(GroupKFold(n_splits=5).split(d[cols], d["target"], groups=d["patient_id"]))
report_opt = skore.evaluate(lgbm_skore, d[cols], d["target"], splitter=cv_skore)
hub_project.put("10_optimized", report_opt)

rmse_rep = float(report_opt.metrics.rmse().iloc[:, 0].values.mean())
mae_rep  = float(report_opt.metrics.mae().iloc[:, 0].values.mean())
print(f"Rapport Hub envoyé : 10_optimized")
print(f"RMSE skore : {rmse_rep:.4f}  MAE : {mae_rep:.4f}")

Output()

Output()

Consult your report at https://skore.probabl.ai/BERNIE 
MADOFF/bobathon-esilv/cross-validations/67191

Rapport Hub envoyé : 10_optimized
RMSE skore : 4.0683  MAE : 3.0044


## 11. Récapitulatif

In [11]:
recap = pd.DataFrame([
    {"Modèle": "tabular_pipeline (skrub)",  "RMSE": 7.53,      "MAE": 5.93},
    {"Modèle": "LightGBM base (NB 11)",     "RMSE": 4.14,      "MAE": 3.08},
    {"Modèle": "LightGBM Optuna (OOF)",     "RMSE": rmse_oof,  "MAE": mae_oof},
    {"Modèle": "+ Stacking Ridge",           "RMSE": rmse_meta, "MAE": mae_meta},
])
display(recap.round(4))

,Modèle,RMSE,MAE
0,tabular_pipeline (skrub),7.5300,5.9300
1,LightGBM base (NB 11),4.1400,3.0800
2,LightGBM Optuna (OOF),4.0711,3.0027
3,+ Stacking Ridge,4.0728,3.0045
